# Auxiliary Management in native FlagQuantum

Complete adapted local Notebook computation: reproduce seed12 and all16 signed amplitudes, synthesize with depth objective and eight-bit ceiling, draw the native circuit and report executed resources. Retained four-bit quantum source counterpart remains separately verified.

All signed amplitudes and the 0.2 error bound retained. Existing exact preparation satisfies this bound; no workspace optimization or approximation compression claim. Native instruction-basis text drawing replaces cloud viewing. No custom CX/U basis, randomized search-seed or cloud optimizer-timeout equivalence; no automatic workspace/depth tradeoff claim.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](workflow.py).

## Native imports for main

In [ ]:
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, prepare_amplitudes, qfunc

In [ ]:
AMPLITUDES = (
    0.245034472820708,
    -0.170081350544172,
    0.167697366233728,
    -0.023905223276522,
    0.343936056437118,
    -0.296692966947323,
    -0.283916724581065,
    0.330582853784989,
    -0.323760293689634,
    0.257046476143475,
    0.153163141682669,
    -0.075162649766642,
    -0.314744872342388,
    -0.249922153985921,
    0.352662065451226,
    -0.015039183255469,
)

## main

In [ ]:
@qfunc
def main(amplitudes: list[float], out: Output[QArray[QBit]]) -> None:
    prepare_amplitudes(amplitudes, 0.2, out)

## build program

In [ ]:
def build_program() -> synth.Program:
    return synth.synthesize(
        main, classical_inputs={"amplitudes": AMPLITUDES}, max_workspace=0
    )

## Native imports for workflow

In [ ]:
import numpy as np
import flagquantum as fq
from flagquantum import synth
from flagquantum.drawer import draw

## source amplitudes

In [ ]:
def source_amplitudes() -> list[float]:
    """Reproduce all sixteen RandomState(12) values without global RNG mutation."""
    amplitudes = 1 - 2 * np.random.RandomState(12).rand(16)
    return [float(value) for value in amplitudes / np.linalg.norm(amplitudes)]

## build workflow

In [ ]:
def build_workflow() -> synth.Program:
    """Select the existing depth objective with the original eight-bit ceiling."""
    return synth.synthesize(
        main,
        classical_inputs={"amplitudes": source_amplitudes()},
        objective="depth",
        max_qubits=8,
        memory_limit_bytes=64 * 1024**2,
    )

## run example

In [ ]:
def run_example() -> None:
    """Generate data, synthesize, draw, execute and report actual resources."""
    program = build_workflow()
    print(draw(program.circuit))
    print(program.report["resources"])
    print(dict(program.probabilities(fq.run(program), registers=("out",))))

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
run_example()

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states


def householder_reference(amplitudes: list[float]) -> fq.Circuit:
    target = torch.tensor(amplitudes, dtype=torch.complex128)
    target = target / torch.linalg.vector_norm(target)
    difference = -target.clone()
    difference[0] += 1
    reflection = torch.eye(target.numel(), dtype=torch.complex128) - 2 * torch.outer(
        difference, difference.conj()
    ) / torch.vdot(difference, difference)
    width = target.numel().bit_length() - 1
    return fq.Circuit(width, dtype=torch.complex128).any(
        *range(width), matrix=reflection
    )


compare_states(build_workflow(), householder_reference(source_amplitudes()), ("out",))
print("Independent full-state comparison: passed")
